In [1]:
import os
import sys

# 현재 작업 디렉토리 기준으로 상위 1단계 폴더를 루트로 설정
current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, '..'))

# sys.path에 추가 (모듈 import용)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# print("프로젝트 루트로 설정된 경로:", project_root)

In [2]:
# 데이터 확인하기 2025.11.21 zero_count_rate > 99% 이상인 컬럼 제거 후 data 7vs3으로 RandomForest 모델 돌리기
import pandas as pd
import numpy  as np

from sklearn.preprocessing import StandardScaler # 데이터 전처리용

import matplotlib.pyplot as plt
import seaborn as sns

# 사용자 모듈 reload용 - 수정 후 즉시 반영을 위해 
import importlib
from utils import preprocessing, user_utils

# 모듈 reload
importlib.reload(preprocessing)
importlib.reload(user_utils)

from utils.preprocessing import load_data, split_features_target, scale_data, data_split, remove_zero_columns2
from utils.user_utils    import get_model_train_eval

c:\ProgramData\anaconda3\envs\ml_dev\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# 데이터 로딩
train, test = load_data()
# 데이터 할당
X_features, y_labels = split_features_target(train) # ID와 TARGET 모두 제거하고 X_train 만들기
X_test     = test.drop(columns=['ID'], axis=1) # test 데이터에서도 ID 제거 

In [5]:
# Data 전처리 1. zero_count_rate이 99%인 컬럼 제거하기 
X_features, X_test = remove_zero_columns2(X_features, X_test)


Train Data Analysis (Threshold: 99.0% )
Train rows: 76,020, columns: 369
Test rows: 75,818, columns: 369

                                   Train Summary (zero_count 내림차순)                                    
                   ColumnName  na_Sum  nUnique          mode  modeFreq modeFreqRate  zero_count zero_count_rate_display
saldo_medio_var13_medio_hace3       0        1      0.000000     76020      100.00%       76020                 100.00%
                     ind_var2       0        1      0.000000     76020      100.00%       76020                 100.00%
        num_reemb_var33_hace3       0        1      0.000000     76020      100.00%       76020                 100.00%
    num_trasp_var17_out_hace3       0        1      0.000000     76020      100.00%       76020                 100.00%
    num_trasp_var33_out_hace3       0        1      0.000000     76020      100.00%       76020                 100.00%
              saldo_var2_ult1       0        1      0.000000     76020

In [6]:
# Data 전처리 2. var3 의 최소값 -99999 를 최빈값으로 변경하기
X_features['var3'] = X_features['var3'].replace(-999999, 2)

In [7]:
# 스케일링
X_train_scaled, X_test_scaled, scaler = scale_data(X_features, X_test)


In [ ]:
# # 레이블의 분포 확인
# cust_cnt = y_labels.value_counts()
# print(cust_cnt) # 1이 불만족 3008명, 만족이 73012

# # 불만족고객의 비율
# cust_rate = cust_cnt[1] / cust_cnt.sum()
# print(f'불만족 고객 비율: {cust_rate:.2f}')

TARGET
0    73012
1     3008
Name: count, dtype: int64
불만족 고객 비율: 0.04


In [ ]:
# first testing model
# XGBoost (xgb) : yjh, kjh
# LightGBM(lgbm) : lsj, ujm
# Random Forest(rf) : lkj, kjh
# Logistic Regression(lr) : yjh, ujm


In [8]:
# 학습/테스트 데이터 분리
X_train, X_val, y_train, y_val = data_split(
  X_features, 
  y_labels,
  size=0.3
)


In [ ]:
# Model 학습, 평가
from sklearn.ensemble import RandomForestClassifier 

model_name = 'RandomForest_99per_basic7v3'

# 기본값으로 우선 RF 해 보자
rf_clf = RandomForestClassifier(
  random_state = 0,
  n_estimators = 100,
  max_depth    = 8, # RF : 애가 핵심이야 Tree 계열이니까~ 약한 Tree로 만들어야해 그래서 max_depth로 자른거
  n_jobs       = -1 # 병렬처리 여부 
)


# 함수 이용
get_model_train_eval(rf_clf, model_name, X_train, X_val, y_train, y_val)
# AUC: 0.8213, 정확도: 0.9604, 정밀도: 0.5000, 재현율: 0.0017, F1: 0.0033

✓ 모델 저장 완료: models\RandomForest_99per_basic7v3.pkl
  파일 크기: 1.55 MB
folder = c:\big20\git\big20-ML-project2-team3\SantanderCS\results
AUC: 0.8159, 정확도: 0.9605, 정밀도: 1.0000, 재현율: 0.0011, F1: 0.0022
오차행렬:
[[21904     0]
 [  901     1]]
실행 시간: 1.3864390850067139


In [10]:
model_name = 'RandomForest_99per_7v3HP_maxDepth10'

# 기본값으로 우선 RF 해 보자
rf_clf = RandomForestClassifier(
  random_state = 0,
  n_estimators = 100,
  max_depth    = 10, # RF : 애가 핵심이야 Tree 계열이니까~ 약한 Tree로 만들어야해 그래서 max_depth로 자른거
  n_jobs       = -1 # 병렬처리 여부 
)


# 함수 이용
get_model_train_eval(rf_clf, model_name, X_train, X_val, y_train, y_val)
# AUC: 0.8252, 정확도: 0.9605, 정밀도: 0.6667, 재현율: 0.0033, F1: 0.0066

✓ 모델 저장 완료: models\RandomForest_99per_7v3HP_maxDepth10.pkl
  파일 크기: 2.92 MB
folder = c:\big20\git\big20-ML-project2-team3\SantanderCS\results
AUC: 0.8234, 정확도: 0.9604, 정밀도: 0.5000, 재현율: 0.0011, F1: 0.0022
오차행렬:
[[21903     1]
 [  901     1]]
실행 시간: 1.5583713054656982


In [ ]:
# 전체적으로 8vs2 보다 7v3 이 더 auc값이 낮게 나옴. 8v2 로 하는게 좋을 듯